## get all kinds of error 

In [15]:
import pandas as pd
import numpy as np

In [16]:
## SETTINGS

In [17]:
## SETTINGS

# data split
# SPLIT = 'train'
# SPLIT = 'val'
SPLIT = 'test'


MODEL_IDX = '7ws4jdfi/checkpoints/'
# MODEL_IDX = 'model_2'
# MODEL_IDX = 'model_3'
# MODEL_IDX = 'model_4'
# MODEL_IDX = 'model_5'



In [18]:
filename = f'./results/elastic_tensor_{MODEL_IDX}_{SPLIT}.json'

df = pd.read_json(filename)

df.head()

,structure,crystal_system,elastic_tensor,elastic_tensor_voigt,elastic_tensor_pred,elastic_tensor_voigt_pred,k_voigt,k_reuss,k_vrh,g_voigt,g_reuss,g_vrh,y_mod,k_voigt_pred,k_reuss_pred,k_vrh_pred,g_voigt_pred,g_reuss_pred,g_vrh_pred,y_mod_pred
5601,"{'@module': 'pymatgen.core.structure', '@class...",orthorhombic,"[[[[188.92258398, 0.0, 0.0], [0.0, 94.6799928,...","[[188.92258398, 94.6799928, 25.63916148, 0.0, ...","[[[[209.2413330078, 2.1175e-06, -3.45000000000...","[[209.2413330078, 96.2538452148, 34.0150909424...",95.434422,77.052374,86.243398,46.700842,37.236466,41.968654,108.333218,98.898249,86.704028,92.801138,49.129728,40.191112,44.660420,115.459659
2108,"{'@module': 'pymatgen.core.structure', '@class...",cubic,"[[[[168.02043234, 0.0, 0.0], [0.0, 117.4076542...","[[168.02043234, 117.4076542, 117.4076542, 0.0,...","[[[[156.5265045166, 1.35e-08, 6.58000000000000...","[[156.5265045166, 142.54737854, 142.5472259521...",134.278580,134.278580,134.278580,20.644909,19.992337,20.318623,58.028948,147.207062,147.207062,147.207062,32.009011,14.378048,23.193530,66.108626
2516,"{'@module': 'pymatgen.core.structure', '@class...",cubic,"[[[[380.44360583, 0.0, 0.0], [0.0, 114.9664217...","[[380.44360583, 114.96642177, 114.96642177, 0....","[[[[348.509552002, -1.232e-06, -6.199e-07], [-...","[[348.509552002, 125.8965454102, 125.896560668...",203.458816,203.458816,203.458816,98.947897,92.041107,95.494502,247.726281,200.100884,200.100884,200.100884,91.520054,88.859901,90.189977,235.228937
7318,"{'@module': 'pymatgen.core.structure', '@class...",tetragonal,"[[[[167.45943092, 0.0, 0.0], [0.0, 39.46690416...","[[167.45943092, 39.46690416, 49.76305771, 0.0,...","[[[[176.9217376709, 3.5000000000000003e-09, 1....","[[176.9217376709, 45.3444328308, 56.2122383118...",86.199172,86.177347,86.188260,53.219639,52.601464,52.910552,131.767788,92.802737,92.802707,92.802722,52.908611,52.084652,52.496631,132.504796
33,"{'@module': 'pymatgen.core.structure', '@class...",cubic,"[[[[354.81570936, 0.0, 0.0], [0.0, 201.0811404...","[[354.81570936, 201.08114046, 201.08114046, 0....","[[[[364.8249816895, -5.9e-09, -8.8000000000000...","[[364.8249816895, 215.1750793457, 215.17507934...",252.325997,252.325997,252.325997,109.432457,102.260492,105.846474,278.585464,265.058380,265.058380,265.058380,117.149438,105.559505,111.354472,293.028402


In [19]:
df.groupby('crystal_system')['crystal_system'].count()

crystal_system
cubic           434
hexagonal       141
monoclinic       54
orthorhombic    139
tetragonal      182
triclinic         4
trigonal         73
Name: crystal_system, dtype: int64

## statistics of target 

In [20]:
prop = ['k_voigt', 'k_reuss', 'k_vrh', 'g_voigt', 'g_reuss', 'g_vrh', 'y_mod']

df[prop].describe()

,k_voigt,k_reuss,k_vrh,g_voigt,g_reuss,g_vrh,y_mod
count,1027.000000,1027.000000,1027.000000,1027.000000,1027.000000,1027.000000,1027.000000
mean,108.668166,106.109120,107.388643,54.616646,47.000969,50.808808,129.805478
std,72.079762,72.160135,71.982747,40.209359,38.290697,38.774724,96.517873
min,5.074422,2.536770,5.074422,2.797444,2.411163,2.625154,6.717135
25%,51.952579,50.365492,51.588833,25.450827,18.867687,22.630672,58.876599
50%,89.753704,87.394631,88.439053,42.726108,35.782944,38.924387,99.250568
75%,155.322199,153.163755,154.116490,74.519067,67.315064,71.116449,180.323499
max,400.983634,399.836465,400.410050,293.212833,260.084365,265.327544,646.989362


In [21]:
crystal_systems = ['cubic','tetragonal', 'hexagonal', 'orthorhombic', 'trigonal', 'monoclinic','triclinic']


def mae(a,b):
    return np.mean(np.abs(np.array(a) - np.array(b))).item()

def mad(x): 
    return np.mean(np.abs(x - np.mean(x))).item()

In [22]:
def filter_outliers(a, b, diff_abs:float):
    a = np.array(a)
    b = np.array(b)

    diff = np.abs(a-b)

    index = np.where(diff<diff_abs)

    return a[index], b[index]

## MAE in tensor 

In [23]:
all_error = {}

In [24]:
x = mae(df['elastic_tensor'].to_list(), df['elastic_tensor_pred'].to_list())
all_error['MAE_tensor'] = x
print(x)

3.16104325109626


In [25]:
x = mae(df['elastic_tensor_voigt'].to_list(), df['elastic_tensor_voigt_pred'].to_list())
all_error['MAE_voigt_matrix'] = x
print(x)

4.344355476836562


## MAE in tensor by crystal system 

In [26]:
all_error['mae_tensor_crystal'] = {'Crystal':[], 'MAE': []}

for crystal in crystal_systems:
    group = df[df['crystal_system']==crystal]
    
    all_error['mae_tensor_crystal']['Crystal'].append(crystal)
    all_error['mae_tensor_crystal']['MAE'].append(mae(group['elastic_tensor'].to_list(), group['elastic_tensor_pred'].to_list()))

all_error['mae_tensor_crystal']

{'Crystal': ['cubic',
  'tetragonal',
  'hexagonal',
  'orthorhombic',
  'trigonal',
  'monoclinic',
  'triclinic'],
 'MAE': [2.8606299019385903,
  3.0991078752634786,
  3.2405123588782683,
  3.2296146009571896,
  4.0084567403221545,
  4.107221982366553,
  5.151101727590433]}

In [27]:
all_error['mae_voigt_crystal'] = {'Crystal':[], 'MAE': []}

for crystal in crystal_systems:
    group = df[df['crystal_system']==crystal]

    all_error['mae_voigt_crystal']['Crystal'].append(crystal)
    all_error['mae_voigt_crystal']['MAE'].append(mae(group['elastic_tensor_voigt'].to_list(), group['elastic_tensor_voigt_pred'].to_list()))

all_error['mae_voigt_crystal']

{'Crystal': ['cubic',
  'tetragonal',
  'hexagonal',
  'orthorhombic',
  'trigonal',
  'monoclinic',
  'triclinic'],
 'MAE': [4.043972343270634,
  4.245172376976602,
  4.536383763690346,
  4.638111631886531,
  4.7681931896210425,
  5.119602418466409,
  6.270861042465973]}

## MAE in elastic property 

In [28]:
all_error['mae_prop'] = {'Property':[], 'MAE': []}

for p in prop:
    all_error['mae_prop']['Property'].append(p)

    a = df[p].to_list()
    b = df[p+'_pred'].to_list()
    if p[:2] in ['k_','g_', 'yo']:
        a,b = filter_outliers(a,b, 100)

    x = mae(a,b)

    all_error['mae_prop']['MAE'].append(x)

all_error['mae_prop'] 

{'Property': ['k_voigt',
  'k_reuss',
  'k_vrh',
  'g_voigt',
  'g_reuss',
  'g_vrh',
  'y_mod'],
 'MAE': [7.301409511955015,
  7.832710609444445,
  7.511140317940507,
  7.7048580399052575,
  9.60976907771908,
  8.548724880753367,
  102.35137326781813]}

## MAE in elastic property, log10 space  

In [29]:
all_error['mae_prop_log10'] = {'Property':[], 'MAE': []}

for p in prop:
    all_error['mae_prop_log10']['Property'].append(p)
    
    tgt = df[p].to_numpy()
    pred = df[p+'_pred'].to_numpy()

    if p[:2] in ['k_','g_', 'yo']:
        tgt,pred = filter_outliers(tgt,pred, 100)

    index = np.where(pred>0.1)
    tgt=tgt[index]
    pred=pred[index]

    x = mae(np.log10(tgt), np.log10(pred))
    
    all_error['mae_prop_log10']['MAE'].append(x)

all_error['mae_prop_log10'] 

{'Property': ['k_voigt',
  'k_reuss',
  'k_vrh',
  'g_voigt',
  'g_reuss',
  'g_vrh',
  'y_mod'],
 'MAE': [0.039779779914718635,
  0.05145526984212527,
  0.043888405096621855,
  0.07109229308347488,
  0.1194933632247431,
  0.08911250437649677,
  0.08459779376784843]}

## MAE in elastic property by crystal system 

In [30]:

all_error['mae_prop_crystal'] = {}

for p in prop:
    
    tmp_d = {'Crystal':[], 'MAE': []}
    
    for crystal in crystal_systems:
        group = df[df['crystal_system']==crystal]

        a = group[p].to_list()
        b = group[p+'_pred'].to_list()
        if p[:2] in ['k_','g_', 'yo']:
            a,b = filter_outliers(a,b, 100)
        x = mae(a,b)

        
        tmp_d['Crystal'].append(crystal)
        tmp_d['MAE'].append(x)
    
    all_error['mae_prop_crystal'][p] = tmp_d

        
all_error['mae_prop_crystal'] 

{'k_voigt': {'Crystal': ['cubic',
   'tetragonal',
   'hexagonal',
   'orthorhombic',
   'trigonal',
   'monoclinic',
   'triclinic'],
  'MAE': [6.7997539155633655,
   7.645451151006044,
   7.749262158151773,
   8.124620751495684,
   7.531995097058904,
   6.297568000935185,
   11.027424261775002]},
 'k_reuss': {'Crystal': ['cubic',
   'tetragonal',
   'hexagonal',
   'orthorhombic',
   'trigonal',
   'monoclinic',
   'triclinic'],
  'MAE': [6.799754328588249,
   7.4604252568281755,
   9.119208826585814,
   8.550501612659712,
   9.533797915028767,
   9.918966885455559,
   7.252776719199998]},
 'k_vrh': {'Crystal': ['cubic',
   'tetragonal',
   'hexagonal',
   'orthorhombic',
   'trigonal',
   'monoclinic',
   'triclinic'],
  'MAE': [6.799754122075116,
   7.643195954397252,
   8.184654865649646,
   8.244325184909354,
   8.310963644645206,
   7.935701278964815,
   9.140100490449997]},
 'g_voigt': {'Crystal': ['cubic',
   'tetragonal',
   'hexagonal',
   'orthorhombic',
   'trigonal',
   '

## MAE/MAD 

In [31]:
all_error['mae/mad'] = {'Property':[], 'MAE/MAE': []}

for p in prop:

    a = df[p].to_list()
    b = df[p+'_pred'].to_list()
    if p[:2] in ['k_','g_', 'yo']:
        a,b = filter_outliers(a,b, 100)

    MAD = mad(a)
    MAE = mae(a,b)
    
    all_error['mae/mad']['Property'].append(p)
    all_error['mae/mad']['MAE/MAE'].append(MAE/MAD)

all_error['mae/mad']


{'Property': ['k_voigt',
  'k_reuss',
  'k_vrh',
  'g_voigt',
  'g_reuss',
  'g_vrh',
  'y_mod'],
 'MAE/MAE': [0.12427500227799507,
  0.13305203806032198,
  0.12805615344914603,
  0.25103412533206976,
  0.326128873129792,
  0.28627677779176547,
  1.3635922636751028]}

## MAE/MAD by crystal system
a single mad for all crystal systems

In [32]:
all_error['mae/mad_prop_crystal'] = {}

for p in prop:
    
    tmp_d = {'Crystal':[], 'MAE/MAD': []}


    a = df[p].to_list()
    b = df[p+'_pred'].to_list()
    if p[:2] in ['k_','g_', 'yo']:
        a,b = filter_outliers(a,b, 100)
    MAD = mad(a)

    for crystal in crystal_systems:
        group = df[df['crystal_system']==crystal]

        a = group[p].to_list()
        b = group[p+'_pred'].to_list()
        if p[:2] in ['k_','g_', 'yo']:
            a,b = filter_outliers(a,b, 100)

        x = mae(a,b)

        tmp_d['Crystal'].append(crystal)
        tmp_d['MAE/MAD'].append(x/MAD)
    
    all_error['mae/mad_prop_crystal'][p] = tmp_d

all_error['mae/mad_prop_crystal']

{'k_voigt': {'Crystal': ['cubic',
   'tetragonal',
   'hexagonal',
   'orthorhombic',
   'trigonal',
   'monoclinic',
   'triclinic'],
  'MAE/MAD': [0.1157364796431171,
   0.1301308271576687,
   0.13189776176507467,
   0.13828662270576023,
   0.12819972723242182,
   0.10718893062779232,
   0.18769433121216086]},
 'k_reuss': {'Crystal': ['cubic',
   'tetragonal',
   'hexagonal',
   'orthorhombic',
   'trigonal',
   'monoclinic',
   'triclinic'],
  'MAE/MAD': [0.11550550209748303,
   0.12672813215144357,
   0.15490541912935277,
   0.14524495065995247,
   0.16194792662457333,
   0.1684906829024687,
   0.12320086521803213]},
 'k_vrh': {'Crystal': ['cubic',
   'tetragonal',
   'hexagonal',
   'orthorhombic',
   'trigonal',
   'monoclinic',
   'triclinic'],
  'MAE/MAD': [0.11592785122028808,
   0.13030754752915522,
   0.13953878839149553,
   0.14055609751315173,
   0.14169220527633208,
   0.13529442104529085,
   0.15582801830369833]},
 'g_voigt': {'Crystal': ['cubic',
   'tetragonal',
   'he

## MAE/MAD by crystal system (separate)
separate mad for each crystal system

In [33]:
all_error['mae/separate_mad_prop_crystal'] = {}

for p in prop:

    tmp_d = {'Crystal':[], 'MAE/MAD': []}

    # a = df[p].to_list()
    # b = df[p+'_pred'].to_list()
    # if p[:2] in ['k_','g_', 'yo']:
    #     a,b = filter_outliers(a,b, 100)
    #
    for crystal in crystal_systems:
        group = df[df['crystal_system']==crystal]

        a = group[p].to_list()
        b = group[p+'_pred'].to_list()
        if p[:2] in ['k_','g_', 'yo']:
            a,b = filter_outliers(a,b, 100)

        MAD = mad(a)
        x = mae(a,b)

        tmp_d['Crystal'].append(crystal)
        tmp_d['MAE/MAD'].append(x/MAD)

    all_error['mae/separate_mad_prop_crystal'][p] = tmp_d

all_error['mae/separate_mad_prop_crystal']

{'k_voigt': {'Crystal': ['cubic',
   'tetragonal',
   'hexagonal',
   'orthorhombic',
   'trigonal',
   'monoclinic',
   'triclinic'],
  'MAE/MAD': [0.11356477541803692,
   0.14295512068135122,
   0.11369966452269453,
   0.133388492130207,
   0.15374361793025598,
   0.1432622061964614,
   0.33945935551081446]},
 'k_reuss': {'Crystal': ['cubic',
   'tetragonal',
   'hexagonal',
   'orthorhombic',
   'trigonal',
   'monoclinic',
   'triclinic'],
  'MAE/MAD': [0.11356478231609217,
   0.14106841432985243,
   0.13421764899952826,
   0.1393629312330783,
   0.1903362435766923,
   0.22751748028952024,
   0.20990897216323468]},
 'k_vrh': {'Crystal': ['cubic',
   'tetragonal',
   'hexagonal',
   'orthorhombic',
   'trigonal',
   'monoclinic',
   'triclinic'],
  'MAE/MAD': [0.113564778867053,
   0.14472607881822527,
   0.12093536794514416,
   0.1348797068413993,
   0.16776292626281544,
   0.181528044253442,
   0.272687150828119]},
 'g_voigt': {'Crystal': ['cubic',
   'tetragonal',
   'hexagonal',

## save to file 

In [36]:
import yaml

with open(f'./results/all_error_{MODEL_IDX}_{SPLIT}.yaml', 'w') as f:
    yaml.dump(all_error, f)